In [1]:
import pandas as pd
import sqlite3

In [2]:
# Create database
conn = sqlite3.connect("output/scraped_data.db")
cursor = conn.cursor()

# Create table
create_tab_statement = """
CREATE TABLE IF NOT EXISTS nharieng(
    "Tỉnh/Thành phố" TEXT,
    "Thành phố/Quận/Huyện/Thị xã" TEXT,
    "Xã/Phường/Thị trấn" TEXT,
    "Đường phố" TEXT,
    "Chi tiết" TEXT,
    "Nguồn thông tin" TEXT,
    "Tình trạng giao dịch" TEXT,
    "Thời điểm giao dịch/rao bán" DATE,
    "Thông tin liên hệ" TEXT,
    "Giá rao bán/giao dịch" INTEGER,
    "Giá ước tính" INTEGER,
    "Loại đơn giá (đ/m2 hoặc đ/m ngang)" TEXT,
    "Đơn giá đất" REAL,
    "Lợi thế kinh doanh" TEXT,
    "Số tầng công trình" REAL,
    "Tổng diện tích sàn" REAL,
    "Đơn giá xây dựng" REAL,
    "Năm xây dựng" INTEGER,
    "Chất lượng còn lại" REAL,
    "Diện tích đất (m2)" REAL,
    "Kích thước mặt tiền (m)" REAL,
    "Kích thước chiều dài (m)" REAL,
    "Số mặt tiền tiếp giáp" INTEGER,
    "Hình dạng" TEXT,
    "Độ rộng ngõ/ngách nhỏ nhất (m)" REAL,
    "Khoảng cách tới trục đường chính (m)" REAL,
    "Mục đích sử dụng đất" TEXT,
    "Yếu tố khác" TEXT,
    "Tọa độ (vĩ độ)" REAL,
    "Tọa độ (kinh độ)" REAL,
    "Hình ảnh của bài đăng" TEXT,
    Web TEXT
);"""

create_unique_index = """
CREATE UNIQUE INDEX IF NOT EXISTS unique_index
ON nharieng(
            "Tỉnh/Thành phố",  
            "Thành phố/Quận/Huyện/Thị xã",  
            "Xã/Phường/Thị trấn",  
            "Đường phố",  
            "Giá rao bán/giao dịch",  
            "Giá ước tính",  
            "Đơn giá đất",  
            "Lợi thế kinh doanh",  
            "Số tầng công trình",  
            "Tổng diện tích sàn",  
            "Đơn giá xây dựng",  
            "Chất lượng còn lại",  
            "Diện tích đất (m2)",  
            "Kích thước mặt tiền (m)",  
            "Kích thước chiều dài (m)",  
            "Số mặt tiền tiếp giáp",  
            "Hình dạng",  
            "Độ rộng ngõ/ngách nhỏ nhất (m)",  
            "Khoảng cách tới trục đường chính (m)",  
            "Mục đích sử dụng đất");
"""

cursor.execute(create_tab_statement)
cursor.execute(create_unique_index)
conn.commit()
conn.close()

In [3]:
import pandas as pd

onehousing = pd.read_excel('output/Onehousing/30.12.2025-06.01.2026.xlsx')
onehousing.columns

Index(['Tỉnh/Thành phố', 'Thành phố/Quận/Huyện/Thị xã', 'Xã/Phường/Thị trấn',
       'Đường phố', 'Nguồn thông tin', 'Tình trạng giao dịch',
       'Thời điểm giao dịch/rao bán', 'Thông tin liên hệ',
       'Giá rao bán/giao dịch', 'Giá ước tính',
       'Loại đơn giá (đ/m2 hoặc đ/m ngang)', 'Đơn giá đất',
       'Số tầng công trình', 'Chất lượng còn lại', 'Đơn giá xây dựng',
       'Diện tích đất (m2)', 'Tổng diện tích sàn', 'Kích thước mặt tiền (m)',
       'Kích thước chiều dài (m)', 'Số mặt tiền tiếp giáp', 'Hình dạng',
       'Độ rộng ngõ/ngách nhỏ nhất (m)',
       'Khoảng cách tới trục đường chính (m)', 'Mục đích sử dụng đất',
       'Hình ảnh của bài đăng', 'Yếu tố khác'],
      dtype='object')

In [4]:
bds_path = 'output/07.11.2025-08.01.2026.xlsx'
onehousing_path =  'output/Onehousing/30.12.2025-06.01.2026.xlsx'

with sqlite3.connect("output/scraped_data.db") as conn:
    bds_df = pd.read_excel(bds_path)
    bds_df['Web'] = 'Batdongsan'
    onehousing_df = pd.read_excel(onehousing_path)
    onehousing_df['Web'] = 'Onehousing'
    print(bds_df.columns)

    bds_df.to_sql('nharieng', conn, if_exists='append', index=False)
    onehousing_df.to_sql('nharieng', conn, if_exists='append', index=False)
    
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM nharieng;")
    n_rows = cursor.fetchone()[0]
    print(n_rows)
    # Drop duplicated values
    # inspect_dup = """SELECT *, COUNT(*) FROM nharieng
    #               GROUP BY  Tỉnh/Thành phố ,  Thành phố/Quận/Huyện/Thị xã ,  Xã/Phường/Thị trấn ,  Đường phố ,  Giá rao bán/giao dịch ,  Giá ước tính ,  Đơn giá đất ,  Lợi thế kinh doanh ,  Số tầng công trình ,  Tổng diện tích sàn ,  Đơn giá xây dựng ,  Chất lượng còn lại ,  Diện tích đất (m2) ,  Kích thước mặt tiền (m) ,  Kích thước chiều dài (m) ,  Số mặt tiền tiếp giáp ,  Hình dạng ,  Độ rộng ngõ/ngách nhỏ nhất (m) ,  Khoảng cách tới trục đường chính (m) ,  Mục đích sử dụng đất
    #               HAVING COUNT(*) > 1"""
    # delete_dup = """DELETE FROM nharieng
    #                 WHERE rowid NOT IN (
    #                 SELECT MIN(rowid)
    #                 FROM nharieng
    #                 GROUP BY "Tỉnh/Thành phố",  
    #                         "Thành phố/Quận/Huyện/Thị xã",  
    #                         "Xã/Phường/Thị trấn",  
    #                         "Đường phố",  
    #                         "Giá rao bán/giao dịch",  
    #                         "Giá ước tính",  
    #                         "Đơn giá đất",  
    #                         "Lợi thế kinh doanh",  
    #                         "Số tầng công trình",  
    #                         "Tổng diện tích sàn",  
    #                         "Đơn giá xây dựng",  
    #                         "Chất lượng còn lại",  
    #                         "Diện tích đất (m2)",  
    #                         "Kích thước mặt tiền (m)",  
    #                         "Kích thước chiều dài (m)",  
    #                         "Số mặt tiền tiếp giáp",  
    #                         "Hình dạng",  
    #                         "Độ rộng ngõ/ngách nhỏ nhất (m)",  
    #                         "Khoảng cách tới trục đường chính (m)",  
    #                         "Mục đích sử dụng đất")"""

Index(['Tỉnh/Thành phố', 'Thành phố/Quận/Huyện/Thị xã', 'Xã/Phường/Thị trấn',
       'Đường phố', 'Chi tiết', 'Nguồn thông tin', 'Tình trạng giao dịch',
       'Thời điểm giao dịch/rao bán', 'Thông tin liên hệ',
       'Giá rao bán/giao dịch', 'Giá ước tính',
       'Loại đơn giá (đ/m2 hoặc đ/m ngang)', 'Đơn giá đất',
       'Lợi thế kinh doanh', 'Số tầng công trình', 'Tổng diện tích sàn',
       'Đơn giá xây dựng', 'Năm xây dựng', 'Chất lượng còn lại',
       'Diện tích đất (m2)', 'Kích thước mặt tiền (m)',
       'Kích thước chiều dài (m)', 'Số mặt tiền tiếp giáp', 'Hình dạng',
       'Độ rộng ngõ/ngách nhỏ nhất (m)',
       'Khoảng cách tới trục đường chính (m)', 'Mục đích sử dụng đất',
       'Yếu tố khác', 'Tọa độ (vĩ độ)', 'Tọa độ (kinh độ)',
       'Hình ảnh của bài đăng', 'Web'],
      dtype='object')
8460


In [5]:
bds_df.shape[0] + onehousing_df.shape[0]

8460

In [ ]:
import sqlite3

new_bds=pd.read_excel(bds_path)

cols = list(new_bds.columns)
placeholders = ",".join(["?"] * len(cols))
quoted_cols = ",".join(f'"{c}"' for c in cols)

sql = f"""
INSERT OR IGNORE INTO nharieng ({quoted_cols})
VALUES ({placeholders})
"""

with sqlite3.connect("output/scraped_data.db") as conn:
    conn.executemany(sql, new_bds.itertuples(index=False, name=None))
    conn.commit()

    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM nharieng;")
    n_rows = cursor.fetchone()[0]


In [7]:
with sqlite3.connect("output/scraped_data.db") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM nharieng;")
    n_rows = cursor.fetchone()[0]
    print(n_rows)

8485


In [9]:
import sqlite3

with sqlite3.connect('output/scraped_data.db') as conn:
    cursor = conn.cursor()
    cursor.execute("""
        SELECT *
        FROM nharieng
        ORDER BY rowid DESC
        LIMIT 25;
    """)
    rows = cursor.fetchall()

for row in rows:
    print(row)


('Thành phố Hà Nội', 'Quận Hà Đông', 'Phường Hà Cầu', 'Đường Hà Cầu', 'Mặt ngõ', 'https://batdongsan.com.vn/ban-nha-rieng-duong-ha-cau-phuong-ha-cau/ban-chinh-chu-mat-ngo-thong-o-to-do-cua-kinh-doanh-sam-uat-mat-tien-5-m-55-m2-11-ty-75-pr44750448', 'Đang rao bán', '29/12/2025', None, 11750000000, 11515000000, 'đ/m2', 202375641.01, 'Tốt', None, 55.0, 8221171.0, None, 0.85, 55.0, 5.0, 11.0, 1, 'Chữ nhật', 5.0, 30.0, 'Đất ở', '- SIÊU PHẨM TRUNG TÂM HÀ ĐÔNG CÁCH PHỐ 30M Ô TÔ ĐỖ CỬA\n- HÀ CẦU MẶT NGÕ THÔNG KINH DOANH NHÀ ĐẸP Ở NGAY\n\n- Địa chỉ: 45A Hà Cầu Võ Thị Sáu Hà Đông\n- Diện tích sổ: 52m² Diện tích sử dụng 55m²\n- Nhà 4 tầng Mặt tiền 5m\n- Ô tô đỗ cửa ngõ thông tứ phía kinh doanh tốt\n- Giá chào: 11.75 tỷ\n- Sổ đỏ chuẩn chính chủ\n\nĐẶC ĐIỂM NỔI BẬT\n- Vị trí quá đỉnh: chỉ 30m ra mặt phố Võ Thị Sáu, trung tâm phường Hà Cầu\n- Ô tô đỗ cửa, phù hợp mở spa văn phòng kinh doanh online tiệm tóc nail\n- Ngõ thông tứ tung giao thương cực mạnh\n- Nhà dân xây cực kiên cố chắc chắn khách xem 

In [10]:
import sqlite3
import pandas as pd

db_path = "output/scraped_data.db"
out_path = "output/nharieng_export.xlsx"

with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query("SELECT * FROM nharieng", conn)

df.to_excel(out_path, index=False)


In [16]:
df[df['Nguồn thông tin'] == "https://batdongsan.com.vn/ban-nha-rieng-duong-nguyen-huu-canh-phuong-22/can-ban-hem-ba-gac-p22-tret-lau-duc-pr44918979"]

,Tỉnh/Thành phố,Thành phố/Quận/Huyện/Thị xã,Xã/Phường/Thị trấn,Đường phố,Chi tiết,Nguồn thông tin,Tình trạng giao dịch,Thời điểm giao dịch/rao bán,Thông tin liên hệ,Giá rao bán/giao dịch,...,Số mặt tiền tiếp giáp,Hình dạng,Độ rộng ngõ/ngách nhỏ nhất (m),Khoảng cách tới trục đường chính (m),Mục đích sử dụng đất,Yếu tố khác,Tọa độ (vĩ độ),Tọa độ (kinh độ),Hình ảnh của bài đăng,Web
52,Thành phố Hồ Chí Minh,Quận Bình Thạnh,Phường 22,Đường Nguyễn Hữu Cảnh,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,29/12/2025,None,6.800000e+09,...,1,Chữ nhật,3.0,10.0,Đất ở,Bình Thạnh\nCần bán nhà hẻm ba gác đường Nguyễ...,10.7935,106.718105,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
8460,Thành phố Hồ Chí Minh,Quận Bình Thạnh,Phường 22,Đường Nguyễn Hữu Cảnh,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,29/12/2025,None,6.800000e+09,...,1,Chữ nhật,3.0,10.0,Đất ở,Bình Thạnh\nCần bán nhà hẻm ba gác đường Nguyễ...,10.7935,106.718105,"[""https://file4.batdongsan.com.vn/crop/600x315...",None


In [17]:
df.iloc[52] == df.iloc[8460]

Tỉnh/Thành phố                           True
Thành phố/Quận/Huyện/Thị xã              True
Xã/Phường/Thị trấn                       True
Đường phố                                True
Chi tiết                                 True
Nguồn thông tin                          True
Tình trạng giao dịch                     True
Thời điểm giao dịch/rao bán              True
Thông tin liên hệ                       False
Giá rao bán/giao dịch                    True
Giá ước tính                             True
Loại đơn giá (đ/m2 hoặc đ/m ngang)       True
Đơn giá đất                              True
Lợi thế kinh doanh                       True
Số tầng công trình                      False
Tổng diện tích sàn                       True
Đơn giá xây dựng                         True
Năm xây dựng                            False
Chất lượng còn lại                       True
Diện tích đất (m2)                       True
Kích thước mặt tiền (m)                  True
Kích thước chiều dài (m)          

In [20]:
df.iloc[[52, 8460]][['Thông tin liên hệ', 'Số tầng công trình', 'Năm xây dựng', 'Web']]

,Thông tin liên hệ,Số tầng công trình,Năm xây dựng,Web
52,None,NaN,None,Batdongsan
8460,None,NaN,None,None


In [21]:
df.iloc[52]['Số tầng công trình'] == df.iloc[8460]['Số tầng công trình']

np.False_

In [24]:
type(df.iloc[8460]['Số tầng công trình'])

numpy.float64